# Photometric capability coverage audit — FoV and limiting magnitude (A3)

ICARE is the canonical resource universe. This notebook asks one narrow,
descriptive question: **after using every source currently available in the
repository, which ICARE photometric instruments still lack a field of view, a
limiting magnitude, or both?**

The result feeds a short expert question list. It is evidence only — it
normalizes nothing, decides nothing, and writes no resource layer.

Sources, in the order they are trusted for each capability:

| Source | Preserved at | Used for |
|---|---|---|
| ICARE / SkyPortal frozen capture | `data/raw/telescopes/icare/…`, `data/interim/telescopes/…` | resource universe, `region` footprint, `sensitivity_data`, filters, location |
| GRANDMA HDR Table 1.1 | `data/raw/reference/grandma/Second_Phd_Thesis____Sarah-7.pdf` → `…/reference/grandma_table.parquet` | FoV, Mlim, filters, Use/Rob |
| GRANDMA GRB campaign guideline (June 2025) | `data/raw/reference/grandma/grb_campaign_guidelines_2025.md` | Mlim + exposure/filter context, restrictions — **never FoV** |
| Expert feedback (Sarah/Camille) | `data/raw/reference/grandma/ICARE_GRANDMA_TELESCOPES_answered.xlsx` | confirmed identities, exclusions, scope |

Two rules constrain everything below: unknown stays `UNKNOWN`, and no FoV is
ever inferred from aperture, telescope name, limiting magnitude, filters or
pointing history.

## 1. Scope and sources

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 250)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/interim/telescopes").is_dir())
CAPTURE_ID = "capture_20260808_071334"
ICARE_DIR = ROOT / "data/interim/telescopes" / CAPTURE_ID
HDR_PARQUET = ROOT / "data/interim/telescopes/reference/grandma_table.parquet"
GRB_MD = ROOT / "data/raw/reference/grandma/grb_campaign_guidelines_2025.md"
EXPERT_XLSX = ROOT / "data/raw/reference/grandma/ICARE_GRANDMA_TELESCOPES_answered.xlsx"
EXPORT_PATH = ROOT / "exports/telescopes/ICARE_photometric_missing_FoV_Mlim.xlsx"

TEL = pd.read_parquet(ICARE_DIR / "telescopes.parquet")
INST = pd.read_parquet(ICARE_DIR / "instruments.parquet")
OBS = pd.read_parquet(ICARE_DIR / "observations.parquet")
HDR = pd.read_parquet(HDR_PARQUET)

In [2]:
def load_grb_guideline() -> pd.DataFrame:
    """Parse the markdown record table (the only pipe-table with 9 columns)."""
    lines = GRB_MD.read_text(encoding="utf-8").splitlines()
    header_idx = next(i for i, ln in enumerate(lines) if ln.startswith("| Telescope / resource |"))
    columns = [c.strip() for c in lines[header_idx].strip().strip("|").split("|")]
    rows = []
    for line in lines[header_idx + 2:]:
        if not line.startswith("|"):
            break
        cells = [c.strip() for c in line.strip().strip("|").split("|")]
        rows.append(dict(zip(columns, cells)))
    frame = pd.DataFrame(rows)
    frame.columns = ["resource", "section", "region", "aperture", "filters", "limiting_magnitude",
                     "trigger_type", "usage_or_status", "restrictions"]
    return frame


GRB = load_grb_guideline()

In [3]:
def load_expert_feedback() -> pd.DataFrame:
    from openpyxl import load_workbook
    workbook = load_workbook(EXPERT_XLSX)
    sheet = workbook["Questions"]
    header = [c.value for c in sheet[1]]
    rows = [dict(zip(header, r)) for r in sheet.iter_rows(min_row=2, values_only=True)]
    return pd.DataFrame(rows)


EXPERT = load_expert_feedback()

In [4]:
print(f"ICARE telescopes           : {len(TEL)}")
print(f"ICARE instruments          : {len(INST)}")
print(f"GRANDMA HDR rows           : {len(HDR)}  "
      f"(Photometry {(HDR['section'] == 'Photometry').sum()}, "
      f"Spectroscopy {(HDR['section'] == 'Spectroscopy').sum()})")
print(f"GRB guideline rows         : {len(GRB)}")
print(f"expert-feedback rows       : {len(EXPERT)}")

ICARE telescopes           : 89
ICARE instruments          : 95
GRANDMA HDR rows           : 39  (Photometry 32, Spectroscopy 7)
GRB guideline rows         : 56
expert-feedback rows       : 11


The GRB guideline states its own limits in the file: it carries no general
FoV column, and its network-level rows (`KNC`, `Skynet`, `SPECULOS incl
ARTEMIS`) are explicitly not safe per-instrument values. Both constraints are
enforced in code below rather than left to reading discipline.

In [5]:
UNKNOWN_TOKENS = {"", "-", "unknown", "tbc", "n/a", "none", "nan"}


def has_content(value) -> bool:
    if value is None:
        return False
    if isinstance(value, float) and np.isnan(value):
        return False
    if isinstance(value, str):
        return value.strip().lower() not in (UNKNOWN_TOKENS | {"[]", "{}"})
    return True

## 2. Photometric instrument classification

Every ICARE instrument is classified from its own structured metadata first
(`type`, `band`, `filters`), with expert feedback applied only where it
explicitly settles scope. An instrument whose structured fields contradict
each other is reported as `SCOPE_AMBIGUOUS` rather than being quietly pushed
into either bucket.

Four expert answers bear directly on scope:

- ShAO-T2m — *"only for spectro so ignore"*;
- GTC/EMIR — *"it is a spectro so ignore"* (leaving GTC/OSIRIS as the photometric instrument);
- generic `TAROT` — *"ignore as we have already TCA/TCH/TRE, TAROT is a network of telescopes"*;
- `GCN` — *"(generic)"*, i.e. not a physical observing resource.

In [6]:
# Expert feedback that establishes scope/exclusion (verbatim answers in the
# answered workbook). Keyed by ICARE telescope name.
EXPERT_SCOPE_EXCLUSIONS = {
    "ShAO-T2m": "expert: 'only for spectro so ignore'",
    "TAROT": "expert: 'ignore as we have already TCA/TCH/TRE TAROT is a network of telescopes'",
    "GCN": "expert: '(generic)' — not a physical observing resource",
}
# GRANDMA HDR itself files these instrument rows under its Spectroscopy section.
HDR_SPECTROSCOPY_INSTRUMENTS = {"BFOSC", "YFOSC", "Spectrograph UAGS", "Spectrograph Canberra"}


def n_filters(raw: str) -> int:
    if not isinstance(raw, str) or raw.strip() in ("", "[]"):
        return 0
    try:
        return len(json.loads(raw))
    except json.JSONDecodeError:
        return 0


def classify_scope(row: pd.Series) -> tuple[str, str]:
    """Return (scope_status, reason) using ICARE structured metadata first."""
    telescope, name = row["telescope.name"], row["name"]
    itype, band = row["type"], str(row["band"]).strip().lower()
    nfilt = n_filters(row["filters"])

    if telescope in EXPERT_SCOPE_EXCLUSIONS:
        reason = EXPERT_SCOPE_EXCLUSIONS[telescope]
        status = "SPECTROSCOPIC_ONLY" if telescope == "ShAO-T2m" else "EXCLUDED_NON_PHYSICAL"
        return status, reason

    if itype == "spectrograph":
        if nfilt == 0:
            return "SPECTROSCOPIC_ONLY", f"ICARE type='{itype}' and no photometric filters registered"
        if name in HDR_SPECTROSCOPY_INSTRUMENTS:
            return ("SPECTROSCOPIC_ONLY",
                    f"ICARE type='{itype}'; GRANDMA HDR also lists '{name}' under its Spectroscopy section")
        return ("SCOPE_AMBIGUOUS",
                f"ICARE type='{itype}' but {nfilt} broadband photometric filters registered "
                f"(structured metadata contradicts itself)")

    if band in ("xray", "x-ray", "gamma"):
        return ("SCOPE_AMBIGUOUS",
                f"ICARE band='{row['band']}' — high-energy imager, outside the optical/IR "
                f"photometry scope of this audit")

    if itype in ("imager", "imaging spectrograph"):
        return "PHOTOMETRIC", f"ICARE type='{itype}', band='{row['band']}', {nfilt} filters registered"

    return "SCOPE_AMBIGUOUS", f"unrecognised ICARE type='{itype}'"


INST = INST.copy()
INST[["scope_status", "scope_reason"]] = INST.apply(
    lambda r: pd.Series(classify_scope(r)), axis=1)

PHOT = INST[INST["scope_status"] == "PHOTOMETRIC"].copy()

In [7]:
scope_counts = INST["scope_status"].value_counts()
print(scope_counts.to_string())
print(f"\ntotal classified: {int(scope_counts.sum())} of {len(INST)} ICARE instruments")

print("\nExcluded — SPECTROSCOPIC_ONLY:")
print(INST.loc[INST["scope_status"] == "SPECTROSCOPIC_ONLY",
               ["id", "name", "telescope.name", "type", "band", "scope_reason"]].to_string(index=False))

print("\nExcluded — EXCLUDED_NON_PHYSICAL (generic/network records, expert-established):")
print(INST.loc[INST["scope_status"] == "EXCLUDED_NON_PHYSICAL",
               ["id", "name", "telescope.name", "type", "band", "scope_reason"]].to_string(index=False))

print("\nSCOPE_AMBIGUOUS — kept out of the capability audit, not discarded:")
print(INST.loc[INST["scope_status"] == "SCOPE_AMBIGUOUS",
               ["id", "name", "telescope.name", "type", "band", "scope_reason"]].to_string(index=False))

scope_status
PHOTOMETRIC              86
SPECTROSCOPIC_ONLY        4
SCOPE_AMBIGUOUS           3
EXCLUDED_NON_PHYSICAL     2

total classified: 95 of 95 ICARE instruments

Excluded — SPECTROSCOPIC_ONLY:
 id                  name telescope.name         type    band                                                                             scope_reason
 15     Spectrograph UAGS       ShAO-T2m spectrograph optical                                                     expert: 'only for spectro so ignore'
 45                 YFOSC       GMG-2.4m spectrograph Optical                          ICARE type='spectrograph' and no photometric filters registered
 14 Spectrograph Canberra       ShAO-T2m spectrograph optical                                                     expert: 'only for spectro so ignore'
 43                 BFOSC Xinglong-2.16m spectrograph Optical ICARE type='spectrograph'; GRANDMA HDR also lists 'BFOSC' under its Spectroscopy section

Excluded — EXCLUDED_NON_PHYSICAL (generic

`GMG-2.4` is the honest ambiguity here: ICARE types it `spectrograph`, yet it
carries eight broadband photometric filters and the GRANDMA HDR lists
`GMG-2.4` in its **Photometry** section (FoV 0.17 x 0.17, Mlim 23). The two
structured sources disagree about what the instrument is, so it is reported
rather than assigned. The two X-ray imagers (`EP-FXT`, SVOM `MXT`) are
imagers but sit outside the optical/IR photometry scope of this audit.

## 3. Cross-source identity mapping

Mappings use several pieces of evidence at once — alias/name, observatory,
aperture, instrument name, and expert confirmation — never name-token
similarity alone. Each mapping records its own basis so the assignment can be
audited row by row.

Two mappings are expert-confirmed rather than computed: `PicduMidi/T1M =
TRoPPIC`, and GTC/OSIRIS as the photometric GTC instrument. GRANDMA HDR row 6
(`BJP/ALI-50 TNOT`) is deliberately left unmapped — the expert answered *"Ali
does not function"*.

In [8]:
# (hdr_row_index, icare_telescope, icare_instrument_or_None, match_basis)
# icare_instrument set only where the HDR row names the instrument itself.
HDR_MAP = [
    (0, "Thai Robotic Telescope - SBO", None, "alias TRT-SBO; aperture 0.70 m; Spring Brook Obs."),
    (1, "Thai National Telescope", None, "Thai Nat Obs.; aperture 2.40 m exact"),
    (2, "Xinglong-TNT", None, "Xinglong Obs.; aperture 0.80 m exact"),
    (3, "Zadko", None, "exact name; aperture 1.00 m"),
    (4, "Xinglong-2.16m", None, "name variant; aperture 2.16 m exact"),
    (5, "GMG-2.4m", None, "name variant; aperture 2.40 m exact"),
    (7, "UBAI/NT-60", None, "exact name; aperture 0.60 m"),
    (8, "UBAI/ST-60", None, "name prefix 'UBAI/ST-60'; aperture 0.60 m"),
    (9, "TAROT/TRE", None, "exact name; La Reunion"),
    (10, "Les-Makes/T60", None, "name variant; La Reunion; aperture 0.60 m"),
    (11, "Terskol/Zeiss-600", None, "name variant; Terskol Obs.; aperture 0.60 m"),
    (12, "ShAO-T60", None, "name variant; Shamakhy Obs.; aperture 0.60 m"),
    (13, "AbAO-T70", None, "name variant; Abastumani; aperture 0.70 m"),
    (14, "AbAO-T150", None, "name variant; Abastumani; aperture 1.5 m"),
    (15, "Lisnyky/AZT-8", None, "name variant; Lisnyky Obs.; aperture 0.70 m"),
    (16, "Lisnyky/Schmidt/Cassegrain", None, "name variant; Lisnyky Obs.; aperture 0.36 m"),
    (17, "KAO", None, "exact name; aperture 1.88 m"),
    (18, "TAROT/TCA", None, "exact name; Calern Obs."),
    (19, "FRAM-CTA-N", None, "name variant; ORM; aperture 0.25 m"),
    (20, "OHP/IRIS", None, "name variant; OHP; aperture 0.50 m"),
    (21, "OAJ/T80", None, "exact name; aperture 0.80 m"),
    (22, "PicduMidi/T1M", None, "EXPERT-CONFIRMED: 'PicduMidi/T1M = TRoPPIC so mentioned as PicduMidi/T1M'"),
    (23, "OHP/T120", None, "name variant; OHP; aperture 1.20 m"),
    (24, "OSN/T150", None, "exact name; aperture 1.50 m"),
    (25, "VIRT", None, "exact name; Etelman Obs."),
    (26, "TAROT/TCH", None, "exact name; La Silla Obs."),
    (27, "OPD/1.6m", None, "Pico dos Dias Obs. = OPD; aperture 1.60 m exact; elevation consistent"),
    (28, "FRAM-Auger", None, "exact name; Auger Obs."),
    (29, "Thai Robotic Telescope - SRO", None, "alias TRT-SRO; aperture 0.70 m; California"),
    (30, "Canada-France-Hawaii Telescope", "CFHT/WIRCAM", "HDR names the instrument: CFHT/WIRCam"),
    (31, "Canada-France-Hawaii Telescope", "CFHT/MEGACAM", "HDR names the instrument: CFHT/MegaCam"),
]
# HDR row 6 (BJP/ALI-50 TNOT) deliberately unmapped: expert answered "Ali does not function".

# GRB guideline mappings: (grb_resource, icare_telescope, scope, match_basis)
GRB_MAP = [
    ("TAROT/TRE (La Réunion)", "TAROT/TRE", "instrument-specific", "exact name"),
    ("Les Makes-60", "Les-Makes/T60", "instrument-specific", "name variant; La Reunion; 0.6 m"),
    ("TRT/GAO (China)", "Thai Robotic Telescope - GAO", "instrument-specific", "alias TRT-GAO"),
    ("TRT/SBO (Australia)", "Thai Robotic Telescope - SBO", "instrument-specific", "alias TRT-SBO"),
    ("NUTTelA-TAO (Kazakhstan)", "NUTTelA-TAO ", "instrument-specific", "exact name (ICARE name has a trailing space)"),
    ("FRAM/CTA-N (La Palma)", "FRAM-CTA-N", "instrument-specific", "name variant; La Palma"),
    ("TAROT/TCA (France)", "TAROT/TCA", "instrument-specific", "exact name"),
    ("OHP/IRiS (France)", "OHP/IRIS", "instrument-specific", "name variant; OHP"),
    ("Abastumani-T150 (Georgia)", "AbAO-T150", "instrument-specific", "Abastumani T150; 1.5 m"),
    ("FRAM/Auger (Argentina)", "FRAM-Auger", "instrument-specific", "name variant"),
    ("TAROT/TCH (Chile)", "TAROT/TCH", "instrument-specific", "exact name"),
    ("VIRT (USA)", "VIRT", "instrument-specific", "exact name"),
    ("TRT/SRO (USA)", "Thai Robotic Telescope - SRO", "instrument-specific", "alias TRT-SRO"),
    ("TRT/CTO (Chile)", "Thai Robotic Telescope - CTO", "instrument-specific", "alias TRT-CTO"),
    ("TNT (China)", "Xinglong-TNT", "instrument-specific", "TNT in China; 0.8 m"),
    ("SHAO/T60 (Azerbaijan)", "ShAO-T60", "instrument-specific", "name variant; 0.6 m"),
    ("UBAI-NT/ST (Uzbekistan)", "UBAI/NT-60", "network-level", "grouped row spanning two ICARE telescopes (NT-60 and ST-60)"),
    ("UBAI-NT/ST (Uzbekistan)", "UBAI/ST-60", "network-level", "grouped row spanning two ICARE telescopes (NT-60 and ST-60)"),
    ("Abastumani-T70 (Georgia)", "AbAO-T70", "instrument-specific", "Abastumani T70; 0.7 m"),
    ("GMG-2m (China)", "GMG-2.4m", "telescope-level", "GMG in China (guideline aperture 2.0 m vs ICARE 2.4 m)"),
    ("Xinglong-2.16 (China)", "Xinglong-2.16m", "instrument-specific", "exact name; 2.16 m"),
    ("HAO1 (Morocco)", "HAO", "instrument-specific", "HAO in Morocco; 0.3 m vs ICARE 0.318 m"),
    ("KAO (Europe)", "KAO", "instrument-specific", "exact name; 2 m"),
    ("Pic du Midi (France)", "PicduMidi/T1M", "instrument-specific", "Pic du Midi 1 m; consistent with expert-confirmed TRoPPIC mapping"),
    ("OPD (Brazil)", "OPD/1.6m", "network-level", "grouped row spanning two ICARE telescopes (OPD 0.6 m and 1.6 m)"),
    ("OPD (Brazil)", "OPD/60cm", "network-level", "grouped row spanning two ICARE telescopes (OPD 0.6 m and 1.6 m)"),
    ("SOAR (4m)", "SOAR/Spartan", "instrument-specific", "SOAR 4 m vs ICARE 4.1 m"),
    ("KNC", "KNC-GEN", "network-level", "GRANDMA amateur network row"),
    ("KNC", "KNC-CDK14inch_112", "network-level", "GRANDMA amateur network row"),
    ("KNC", "KNC-FRANCE", "network-level", "GRANDMA amateur network row"),
    ("KNC", "KNC-T-BGO", "network-level", "GRANDMA amateur network row"),
    ("KNC", "KNC-T-BRO", "network-level", "GRANDMA amateur network row"),
    ("KNC", "T72-iTelescope_117", "network-level", "expert: 'T72-iTelescope_117 (KNC)'"),
    ("Skynet", "SKYNET", "network-level", "external network row"),
    ("ASTEP (Antarctica)", "ASTEP", "instrument-specific", "exact name; Antarctica"),
    ("YAPTH (China)", "Yaoan High Precision Telescope", "instrument-specific", "guideline spelling 'YAPTH' = YAHPT; 0.8 m"),
    ("C2PU (France)", "C2PU/Omicron", "instrument-specific", "C2PU in France; 1 m"),
    ("NAO-2m (Bulgaria)", "NAO-2m", "instrument-specific", "exact name; 2 m"),
    ("GTC", "Gran Telescopio Canarias", "instrument-specific", "GTC 10.4 m; single photometric instrument GTC/OSIRIS"),
    ("Euler (Chile)", "Euler", "instrument-specific", "exact name; Chile"),
    ("SPECULOS incl ARTEMIS", "SNO", "network-level", "'8 x 1 m' multi-telescope row; ICARE instrument is Artemis"),
    ("CFHT", "Canada-France-Hawaii Telescope", "telescope-level", "CFHT 3.6 m; two photometric instruments in ICARE"),
    ("OST CDK (Germany)", "OST-CDK", "instrument-specific", "exact name; Germany"),
]

In [9]:
mapped_hdr = {t for _, t, _, _ in HDR_MAP}
mapped_grb = {t for _, t, _, _ in GRB_MAP}
print(f"GRANDMA HDR photometry rows mapped to an ICARE telescope : {len(HDR_MAP)} of "
      f"{(HDR['section'] == 'Photometry').sum()}")
print(f"GRB guideline rows mapped to an ICARE telescope          : "
      f"{len({r for r, _, _, _ in GRB_MAP})} of {len(GRB)}")
print(f"distinct ICARE telescopes reached by at least one source : "
      f"{len(mapped_hdr | mapped_grb)} of {len(TEL)}")

scope_mix = pd.Series([s for _, _, s, _ in GRB_MAP]).value_counts()
print("\nGRB guideline mapping scope:")
print(scope_mix.to_string())

GRANDMA HDR photometry rows mapped to an ICARE telescope : 31 of 32
GRB guideline rows mapped to an ICARE telescope          : 36 of 56
distinct ICARE telescopes reached by at least one source : 51 of 89

GRB guideline mapping scope:
instrument-specific    29
network-level          12
telescope-level         2


## 4. FoV coverage

Priority is fixed: a registered ICARE footprint wins; otherwise a GRANDMA HDR
FoV, but only where it can be attached to one photometric instrument. The GRB
guideline is never consulted for FoV.

One ICARE detail matters here. The `has_region` flag under-reports: two
instruments carry real footprint geometry while the flag reads `False`, so the
`region` payload itself is parsed rather than the flag.

In [10]:
region_present = INST["region"].notna()
print(f"instruments with has_region == True : {int(INST['has_region'].sum())}")
print(f"instruments with region payload      : {int(region_present.sum())}")
disagree = INST.loc[region_present & ~INST["has_region"].astype(bool),
                    ["id", "name", "telescope.name", "has_region", "region_summary"]]
print("\nflag/payload disagreement (payload is used):")
print(disagree.to_string(index=False))

instruments with has_region == True : 17
instruments with region payload      : 19

flag/payload disagreement (payload is used):
 id          name telescope.name  has_region region_summary
 26       MMATest            ZTF       False               
 61 Les-Makes/T60  Les-Makes/T60       False               


In [11]:
def parse_fov_from_region(region_text: str) -> str | None:
    """Derive an angular FoV representation from a registered DS9 footprint.

    Only geometry that is unambiguous is converted; anything else returns None
    so it can be reported rather than guessed at.
    """
    if not isinstance(region_text, str) or not region_text.strip():
        return None
    box = re.search(r"box\(\s*[-\d.]+\s*,\s*[-\d.]+\s*,\s*([\d.]+)\s*,\s*([\d.]+)", region_text)
    if box:
        return f"{float(box.group(1)):g} x {float(box.group(2)):g} deg (box)"
    circle = re.search(r"circle\(\s*[-\d.]+\s*,\s*[-\d.]+\s*,\s*([\d.]+)", region_text)
    if circle:
        return f"radius {float(circle.group(1)):g} deg (circle)"
    polygons = re.findall(r"polygon\(", region_text)
    if polygons:
        return f"{len(polygons)} polygon footprint(s) — no single width/height derivable"
    return None

In [12]:
# Near-duplicate ICARE telescope records the expert flagged in the answered workbook
# ("strange as there are two telescopes with the same name ?", "we have two ZTF ...").
DUPLICATE_HINTS = {
    "Fraunhofer Telescope at Wendelstein Observatory":
        "ICARE also holds a near-duplicate record 'Fraunhofer 2m Telescope' (instrument "
        "'Wendelstein 3KK') whose FoV is registered as 0.117 x 0.117 deg.",
    "Fraunhofer 2m Telescope":
        "ICARE also holds a near-duplicate record 'Fraunhofer Telescope at Wendelstein "
        "Observatory' (instrument '3KK') with no registered FoV.",
    "ZTF": "ICARE holds two instruments on this telescope record (ZTF and MMATest).",
}

## 5. Mlim capability coverage, and 6. empirical limmag evidence

Capability Mlim keeps its original context — magnitude, filter, exposure and
any `?`/`TBC`/`tentative` marker — and its provenance. Network-level values
are labelled `GROUP_LEVEL_ONLY` instead of being copied onto each member
instrument.

Historical `observation.limmag` is computed separately and never promoted to
capability Mlim: an instrument can have empirical evidence and still be
`capability_mlim_status = UNKNOWN`. No sensitivity model is fitted here.

The matrix below is built in one pass so that every capability cell carries
its status, value, source, scope and match basis.

In [13]:
def build_matrix() -> pd.DataFrame:
    """One row per PHOTOMETRIC ICARE instrument, with capability provenance."""
    hdr_by_index = {int(r["row_index_in_source"]): r for _, r in HDR.iterrows()}
    grb_by_resource = {r["resource"]: r for _, r in GRB.iterrows()}
    phot_per_telescope = PHOT.groupby("telescope.name")["id"].count().to_dict()

    # ---- resolve the HDR mapping down to individual photometric instruments
    hdr_for_instrument: dict[int, dict] = {}
    hdr_unresolved_telescopes: list[tuple[int, str, str]] = []
    for hdr_index, telescope, instrument, basis in HDR_MAP:
        candidates = PHOT[PHOT["telescope.name"] == telescope]
        if instrument is not None:
            target = candidates[candidates["name"] == instrument]
            scope = "instrument-specific"
        elif len(candidates) == 1:
            target = candidates
            scope = "instrument-specific"
        else:
            if len(candidates) > 1:
                hdr_unresolved_telescopes.append((hdr_index, telescope, basis))
            continue
        if target.empty:
            continue
        hdr_for_instrument[int(target.iloc[0]["id"])] = {
            "hdr_index": hdr_index, "row": hdr_by_index[hdr_index], "basis": basis, "scope": scope}

    # ---- resolve the GRB-guideline mapping down to individual instruments
    grb_for_instrument: dict[int, list[dict]] = {}
    for resource, telescope, scope, basis in GRB_MAP:
        candidates = PHOT[PHOT["telescope.name"] == telescope]
        if candidates.empty:
            continue
        effective_scope = scope
        if scope == "telescope-level" and phot_per_telescope.get(telescope, 0) > 1:
            effective_scope = "telescope-level"  # cannot be attributed to one instrument
        for _, candidate in candidates.iterrows():
            grb_for_instrument.setdefault(int(candidate["id"]), []).append({
                "resource": resource, "row": grb_by_resource[resource],
                "scope": effective_scope, "basis": basis})

    # ---- empirical limmag, per instrument
    empirical: dict[int, dict] = {}
    for instrument_id, group in OBS.groupby("instrument_id"):
        limmag = group["limmag"].dropna()
        if limmag.empty:
            continue
        filters = sorted(group["filt"].dropna().unique().tolist())
        exposures = group["exposure_time"].dropna()
        exposure_text = (f"{exposures.min():g}-{exposures.max():g} s" if not exposures.empty else "n/a")
        empirical[int(instrument_id)] = {
            "n": int(len(group)),
            "filters": ", ".join(filters),
            "exposure": exposure_text,
            "summary": (f"limmag {limmag.min():g}-{limmag.max():g} (median {limmag.median():g}) "
                        f"in {', '.join(filters)} at {exposure_text}"),
        }

    sensitivity_columns = [c for c in INST.columns if c.startswith("sensitivity_data.")]
    records = []
    for _, instrument in PHOT.iterrows():
        instrument_id = int(instrument["id"])
        telescope_name = instrument["telescope.name"]
        telescope = TEL[TEL["name"] == telescope_name].iloc[0]
        hdr_hit = hdr_for_instrument.get(instrument_id)
        grb_hits = grb_for_instrument.get(instrument_id, [])

        # ---------------- FoV ----------------
        fov_status = fov_value = fov_source = fov_scope = fov_basis = None
        derived = parse_fov_from_region(instrument["region"])
        if derived:
            # Rule 5A: a registered ICARE footprint IS the field of view. A mosaic of
            # polygons has no single width/height, so the geometry is reported as-is
            # rather than reduced to a made-up number.
            fov_status, fov_value = "KNOWN", derived
            fov_source, fov_scope = "ICARE (derived_from_registered_footprint)", "instrument-specific"
            fov_basis = "ICARE instrument region/footprint registered on this instrument"
        elif hdr_hit is not None and has_content(hdr_hit["row"]["fov_deg"]):
            raw = hdr_hit["row"]["fov_deg"]
            fov_status, fov_value = "KNOWN", raw.replace("ˆ", "x")
            fov_source = f"GRANDMA HDR Table 1.1 row {hdr_hit['hdr_index']} ({hdr_hit['row']['telescope_name']})"
            fov_scope, fov_basis = hdr_hit["scope"], hdr_hit["basis"]
        else:
            fov_status = "UNKNOWN"

        # ---------------- capability Mlim ----------------
        mlim_status = mlim_value = mlim_source = mlim_scope = mlim_basis = None
        icare_sensitivity = {c.split(".", 1)[1]: instrument[c] for c in sensitivity_columns
                             if has_content(instrument[c])}
        if icare_sensitivity:
            parts = {}
            for key, value in icare_sensitivity.items():
                band_name, field = key.rsplit(".", 1)
                parts.setdefault(band_name, {})[field] = value
            rendered = []
            for band_name, fields in parts.items():
                rendered.append(
                    f"{fields.get('limiting_magnitude')} mag in {band_name} "
                    f"({fields.get('exposure_time'):g} s, magsys {fields.get('magsys')}, "
                    f"zeropoint {fields.get('zeropoint')})")
            mlim_status, mlim_value = "KNOWN", "; ".join(rendered)
            mlim_source, mlim_scope = "ICARE sensitivity_data", "instrument-specific"
            mlim_basis = "registered directly on this ICARE instrument"
        else:
            specific = [h for h in grb_hits
                        if h["scope"] == "instrument-specific" and has_content(h["row"]["limiting_magnitude"])]
            grouped = [h for h in grb_hits
                       if h["scope"] != "instrument-specific" and has_content(h["row"]["limiting_magnitude"])]
            if specific:
                hit = specific[0]
                mlim_status = "KNOWN"
                mlim_value = (f"{hit['row']['limiting_magnitude']} "
                              f"(filters: {hit['row']['filters']})")
                mlim_source = f"GRANDMA GRB guideline (June 2025), row '{hit['resource']}'"
                mlim_scope, mlim_basis = "instrument-specific", hit["basis"]
            elif hdr_hit is not None and has_content(hdr_hit["row"]["mlim"]):
                mlim_status = "KNOWN"
                mlim_value = (f"{hdr_hit['row']['mlim']} (typical max in <1 h; "
                              f"filters: {hdr_hit['row']['filter']})")
                mlim_source = (f"GRANDMA HDR Table 1.1 row {hdr_hit['hdr_index']} "
                               f"({hdr_hit['row']['telescope_name']})")
                mlim_scope, mlim_basis = hdr_hit["scope"], hdr_hit["basis"]
            elif grouped:
                hit = grouped[0]
                mlim_status = "GROUP_LEVEL_ONLY"
                mlim_value = (f"{hit['row']['limiting_magnitude']} "
                              f"(filters: {hit['row']['filters']})")
                mlim_source = f"GRANDMA GRB guideline (June 2025), row '{hit['resource']}'"
                mlim_scope, mlim_basis = hit["scope"], hit["basis"]
            else:
                mlim_status = "UNKNOWN"

        # second, independent evidence worth recording alongside the primary value
        secondary = []
        if mlim_status == "KNOWN" and mlim_source and "GRB guideline" in mlim_source \
                and hdr_hit is not None and has_content(hdr_hit["row"]["mlim"]):
            secondary.append(f"GRANDMA HDR row {hdr_hit['hdr_index']}: {hdr_hit['row']['mlim']}")
        if mlim_status == "KNOWN" and mlim_source == "ICARE sensitivity_data":
            for hit in grb_hits:
                if has_content(hit["row"]["limiting_magnitude"]):
                    secondary.append(f"GRB guideline '{hit['resource']}': {hit['row']['limiting_magnitude']}")
            if hdr_hit is not None and has_content(hdr_hit["row"]["mlim"]):
                secondary.append(f"GRANDMA HDR row {hdr_hit['hdr_index']}: {hdr_hit['row']['mlim']}")

        # ---------------- secondary coverage ----------------
        filter_count = n_filters(instrument["filters"])
        filters_status = "KNOWN" if filter_count else "UNKNOWN"
        filters_value = instrument["filters"] if filter_count else ""

        if not bool(telescope["fixed_location"]):
            location_status = "N/A"
        elif pd.notna(telescope["lat"]) and pd.notna(telescope["lon"]):
            location_status = "KNOWN"
        else:
            location_status = "UNKNOWN"

        restriction_bits, restriction_sources = [], []
        for hit in grb_hits:
            row = hit["row"]
            if has_content(row["usage_or_status"]) or has_content(row["restrictions"]):
                restriction_bits.append(
                    f"{row['usage_or_status']} ({row['trigger_type']}) — {row['restrictions']}")
                restriction_sources.append(
                    f"GRB guideline '{hit['resource']}' [{hit['scope']}]")
        if hdr_hit is not None:
            restriction_bits.append(
                f"HDR Use={hdr_hit['row']['use']}, Rob={hdr_hit['row']['rob']}")
            restriction_sources.append(f"GRANDMA HDR row {hdr_hit['hdr_index']}")
        if restriction_bits:
            only_group = grb_hits and all(h["scope"] != "instrument-specific" for h in grb_hits) \
                and hdr_hit is None
            restrictions_status = "PARTIAL" if only_group else "KNOWN"
        else:
            restrictions_status = "UNKNOWN"

        empirical_hit = empirical.get(instrument_id)
        missing_fov = fov_status in ("UNKNOWN", "PARTIAL_SCOPE_UNRESOLVED")
        missing_mlim = mlim_status in ("UNKNOWN", "GROUP_LEVEL_ONLY")
        if missing_fov and missing_mlim:
            missing_category = "BOTH"
        elif missing_fov:
            missing_category = "FOV_ONLY"
        elif missing_mlim:
            missing_category = "MLIM_ONLY"
        else:
            missing_category = "COMPLETE"

        records.append({
            "telescope_id": int(instrument["telescope_id"]),
            "telescope_name": telescope_name,
            "instrument_id": instrument_id,
            "instrument_name": instrument["name"],
            "instrument_type": instrument["type"],
            "band": instrument["band"],
            "scope_status": instrument["scope_status"],
            "filters_status": filters_status,
            "filters_value": filters_value,
            "filters_source": "ICARE" if filter_count else "",
            "location_status": location_status,
            "latitude": telescope["lat"], "longitude": telescope["lon"],
            "elevation": telescope["elevation"],
            "fov_status": fov_status,
            "fov_value_raw": fov_value or "",
            "fov_source": fov_source or "",
            "fov_scope": fov_scope or "",
            "fov_match_basis": fov_basis or "",
            "capability_mlim_status": mlim_status,
            "capability_mlim_value_raw": mlim_value or "",
            "capability_mlim_source": mlim_source or "",
            "capability_mlim_scope": mlim_scope or "",
            "capability_mlim_match_basis": mlim_basis or "",
            "capability_mlim_secondary_evidence": "; ".join(secondary),
            "empirical_limmag_available": "YES" if empirical_hit else "NO",
            "empirical_limmag_n": empirical_hit["n"] if empirical_hit else 0,
            "empirical_limmag_summary": empirical_hit["summary"] if empirical_hit else "",
            "empirical_limmag_filters": empirical_hit["filters"] if empirical_hit else "",
            "empirical_limmag_exposure": empirical_hit["exposure"] if empirical_hit else "",
            "restrictions_status": restrictions_status,
            "restrictions_summary": " | ".join(restriction_bits),
            "restrictions_source": " | ".join(restriction_sources),
            "missing_fov": missing_fov,
            "missing_mlim": missing_mlim,
            "missing_category": missing_category,
        })

    matrix = pd.DataFrame(records).sort_values(["telescope_name", "instrument_name"])
    matrix.attrs["hdr_unresolved_telescopes"] = hdr_unresolved_telescopes
    return matrix

In [14]:
MATRIX = build_matrix()
print(f"coverage matrix rows: {len(MATRIX)}  (one per PHOTOMETRIC ICARE instrument)")
print(f"unique instrument ids: {MATRIX['instrument_id'].nunique()}")

print("\nFoV status:")
print(MATRIX["fov_status"].value_counts().to_string())
print("\nFoV provenance, where KNOWN:")
print(MATRIX.loc[MATRIX["fov_status"] == "KNOWN", "fov_source"]
      .str.split(" row").str[0].value_counts().to_string())

print("\ncapability Mlim status:")
print(MATRIX["capability_mlim_status"].value_counts().to_string())
print("\ncapability Mlim provenance, where KNOWN:")
print(MATRIX.loc[MATRIX["capability_mlim_status"] == "KNOWN", "capability_mlim_source"]
      .str.split(" row").str[0].str.split(",").str[0].value_counts().to_string())

coverage matrix rows: 86  (one per PHOTOMETRIC ICARE instrument)
unique instrument ids: 86

FoV status:
fov_status
UNKNOWN    43
KNOWN      43

FoV provenance, where KNOWN:
fov_source
GRANDMA HDR Table 1.1                        24
ICARE (derived_from_registered_footprint)    19

capability Mlim status:


capability_mlim_status
KNOWN               41
UNKNOWN             36
GROUP_LEVEL_ONLY     9

capability Mlim provenance, where KNOWN:
capability_mlim_source
GRANDMA GRB guideline (June 2025)    26
GRANDMA HDR Table 1.1                13
ICARE sensitivity_data                2


In [15]:
print("GROUP_LEVEL_ONLY Mlim — network values deliberately not attributed to the instrument:")
print(MATRIX.loc[MATRIX["capability_mlim_status"] == "GROUP_LEVEL_ONLY",
                 ["telescope_name", "instrument_name", "capability_mlim_value_raw",
                  "capability_mlim_scope"]].to_string(index=False))

print("\ninstruments carrying two independent Mlim evidences:")
both_evidence = MATRIX[MATRIX["capability_mlim_secondary_evidence"] != ""]
print(both_evidence[["telescope_name", "instrument_name", "capability_mlim_value_raw",
                     "capability_mlim_secondary_evidence"]].to_string(index=False))

GROUP_LEVEL_ONLY Mlim — network values deliberately not attributed to the instrument:
    telescope_name instrument_name                                  capability_mlim_value_raw capability_mlim_scope
 KNC-CDK14inch_112 PlaneWave F7.15                     15-20 mag (filters: BVRI, griz, Green)         network-level
        KNC-FRANCE      KNC-FR-CAM                     15-20 mag (filters: BVRI, griz, Green)         network-level
           KNC-GEN             KNC                     15-20 mag (filters: BVRI, griz, Green)         network-level
         KNC-T-BGO           T-BGO                     15-20 mag (filters: BVRI, griz, Green)         network-level
         KNC-T-BRO           T-BRO                     15-20 mag (filters: BVRI, griz, Green)         network-level
          OPD/60cm        OPD/60cm 22 and 23 mag in 1 h for sure (filters: CUBVRcIc and griz)         network-level
            SKYNET   SKYNET/Prompt                        19 mag (filters: BVRI, griz, Green)         

In [16]:
empirical = MATRIX[MATRIX["empirical_limmag_available"] == "YES"]
print(f"ICARE instruments with historical observations : {OBS['instrument_id'].nunique()}")
print(f"photometric instruments with empirical limmag  : {len(empirical)}")
print(empirical[["telescope_name", "instrument_name", "empirical_limmag_n",
                 "empirical_limmag_filters", "empirical_limmag_exposure",
                 "empirical_limmag_summary", "capability_mlim_status"]].to_string(index=False))

promoted = empirical[(empirical["empirical_limmag_available"] == "YES")
                     & (empirical["capability_mlim_source"].str.contains("observation", case=False))]
print(f"\nempirical limmag promoted to capability Mlim: {len(promoted)}  "
      f"(must be 0 — historical limmag is not a nominal capability)")

ICARE instruments with historical observations : 3
photometric instruments with empirical limmag  : 3
telescope_name instrument_name  empirical_limmag_n empirical_limmag_filters empirical_limmag_exposure                                    empirical_limmag_summary capability_mlim_status
    FRAM-Auger      FRAM-Auger                  40                 bessellr                 120-120 s limmag 12.36-15.69 (median 14.545) in bessellr at 120-120 s                  KNOWN
    FRAM-CTA-N      FRAM-CTA-N                  46                 bessellr                 120-120 s limmag 14.34-17.54 (median 17.215) in bessellr at 120-120 s                  KNOWN
     TAROT/TRE       TAROT/TRE                   7                ps1::open                 180-180 s          limmag 18-18 (median 18) in ps1::open at 180-180 s                  KNOWN

empirical limmag promoted to capability Mlim: 0  (must be 0 — historical limmag is not a nominal capability)


## 7. Secondary coverage and the missing-capability matrix

Filters, location and operational restrictions are recorded but kept out of
the missing-capability decision, which turns on FoV and Mlim only. Telescopes
whose `fixed_location` is `False` — space missions and network records — are
`N/A` for coordinates, not `UNKNOWN`.

For the expert list, `PARTIAL_SCOPE_UNRESOLVED` counts as missing FoV and
`GROUP_LEVEL_ONLY` counts as missing Mlim: what is needed is an
instrument-specific, usable value.

In [17]:
print("filters status:")
print(MATRIX["filters_status"].value_counts().to_string())
print("\nlocation status:")
print(MATRIX["location_status"].value_counts().to_string())
print("  N/A (space mission or network record, coordinates not applicable):",
      sorted(MATRIX.loc[MATRIX["location_status"] == "N/A", "telescope_name"].tolist()))
print("\nrestrictions status:")
print(MATRIX["restrictions_status"].value_counts().to_string())
print("  PARTIAL (network-level evidence only):",
      sorted(MATRIX.loc[MATRIX["restrictions_status"] == "PARTIAL", "telescope_name"].tolist()))

filters status:
filters_status
KNOWN    86

location status:
location_status
KNOWN    81
N/A       5
  N/A (space mission or network record, coordinates not applicable): ['Gaia', 'KNC-GEN', 'SVOM', 'Swift', 'Thai Robotic Telescope']

restrictions status:
restrictions_status
KNOWN      42
UNKNOWN    35
PARTIAL     9
  PARTIAL (network-level evidence only): ['KNC-CDK14inch_112', 'KNC-FRANCE', 'KNC-GEN', 'KNC-T-BGO', 'KNC-T-BRO', 'OPD/60cm', 'SKYNET', 'SNO', 'T72-iTelescope_117']


In [18]:
counts = MATRIX["missing_category"].value_counts()
for category in ["COMPLETE", "BOTH", "FOV_ONLY", "MLIM_ONLY"]:
    print(f"{category:10s}: {int(counts.get(category, 0))}")
print(f"{'total':10s}: {int(counts.sum())}  (photometric instruments: {len(MATRIX)})")

for category in ["BOTH", "FOV_ONLY", "MLIM_ONLY"]:
    subset = MATRIX[MATRIX["missing_category"] == category]
    print(f"\n--- {category} ({len(subset)}) ---")
    print(subset[["telescope_name", "instrument_name"]].to_string(index=False))

COMPLETE  : 30
BOTH      : 32
FOV_ONLY  : 11
MLIM_ONLY : 13
total     : 86  (photometric instruments: 86)

--- BOTH (32) ---
                                 telescope_name instrument_name
  Asteroid Terrestrial-impact Last Alert System           ATLAS
                                      CAHA-3.5m       CAHA-3.5m
                                     CAHA-CAFOS           CAFOS
Fraunhofer Telescope at Wendelstein Observatory             3KK
                                           Gaia            Gaia
                                   Gemini North            GMOS
                                           KAIT            KAIT
                                        KNC-GEN             KNC
                                     Keck I 10m            LRIS
                                       LCO-SAAO        LCO-SAAO
                                     LCO_0m4-39      LCO_0m4-39
                            Liverpool Telescope             IOO
                                           

## 8. Expert-question candidates

The workbook below carries only photometric instruments missing FoV and/or
Mlim, sorted most-incomplete first. It states what we already hold so the
supervisors do not re-derive it, and asks a single question per row: where
else can the missing capability be obtained.

In [19]:
MISSING = MATRIX[MATRIX["missing_fov"] | MATRIX["missing_mlim"]].copy()


def missing_label(row: pd.Series) -> str:
    if row["missing_fov"] and row["missing_mlim"]:
        return "FoV + Mlim"
    return "FoV" if row["missing_fov"] else "Mlim"


QUESTION_BY_LABEL = {
    "FoV + Mlim": ("Is there a source where we can obtain the FoV and limiting "
                   "magnitude/sensitivity for this photometric instrument?"),
    "FoV": "Is there a source where we can obtain the FoV for this photometric instrument?",
    "Mlim": ("Is there a source where we can obtain the limiting magnitude/sensitivity "
             "for this photometric instrument?"),
}


def already_have(row: pd.Series) -> str:
    bits = []
    if row["filters_status"] == "KNOWN":
        filters = ", ".join(json.loads(row["filters_value"]))
        bits.append(f"Filters: {filters} (ICARE).")
    if not row["missing_fov"]:
        bits.append(f"FoV: {row['fov_value_raw']} from {row['fov_source']}.")
    elif row["fov_status"] == "PARTIAL_SCOPE_UNRESOLVED":
        bits.append(f"FoV only at telescope level, not attributable to this instrument: "
                    f"{row['fov_value_raw']}.")
    else:
        bits.append("FoV not found in any available source.")
    if not row["missing_mlim"]:
        bits.append(f"Mlim: {row['capability_mlim_value_raw']} from {row['capability_mlim_source']}.")
    elif row["capability_mlim_status"] == "GROUP_LEVEL_ONLY":
        bits.append(f"Only a network-level value exists ({row['capability_mlim_value_raw']}, "
                    f"{row['capability_mlim_source']}), which cannot be assumed for this instrument.")
    else:
        bits.append("No instrument-specific limiting magnitude found.")
    if row["empirical_limmag_available"] == "YES":
        bits.append(f"Historical observations only: {row['empirical_limmag_summary']} "
                    f"({row['empirical_limmag_n']} observations) — not a nominal capability.")
    if row["telescope_name"] in DUPLICATE_HINTS:
        bits.append(DUPLICATE_HINTS[row["telescope_name"]])
    return " ".join(bits)


MISSING["Missing"] = MISSING.apply(missing_label, axis=1)
MISSING["What we already have"] = MISSING.apply(already_have, axis=1)
MISSING["Question"] = MISSING["Missing"].map(QUESTION_BY_LABEL)

ORDER = {"FoV + Mlim": 0, "FoV": 1, "Mlim": 2}
MISSING = MISSING.sort_values(
    by=["Missing", "telescope_name", "instrument_name"],
    key=lambda s: s.map(ORDER) if s.name == "Missing" else s.str.lower())

EXPERT_SHEET = MISSING.rename(columns={"telescope_name": "ICARE Telescope",
                                       "instrument_name": "ICARE Instrument"})
EXPERT_SHEET = EXPERT_SHEET[["ICARE Telescope", "ICARE Instrument", "Missing",
                             "What we already have", "Question"]].copy()
EXPERT_SHEET["Answer"] = ""
EXPERT_SHEET["Comment"] = ""

print(f"expert-question rows: {len(EXPERT_SHEET)}")
print(EXPERT_SHEET["Missing"].value_counts().to_string())
print()
print(EXPERT_SHEET[["ICARE Telescope", "ICARE Instrument", "Missing"]].to_string(index=False))

expert-question rows: 56
Missing
FoV + Mlim    32
Mlim          13
FoV           11

                                ICARE Telescope ICARE Instrument    Missing
  Asteroid Terrestrial-impact Last Alert System            ATLAS FoV + Mlim
                                      CAHA-3.5m        CAHA-3.5m FoV + Mlim
                                     CAHA-CAFOS            CAFOS FoV + Mlim
Fraunhofer Telescope at Wendelstein Observatory              3KK FoV + Mlim
                                           Gaia             Gaia FoV + Mlim
                                   Gemini North             GMOS FoV + Mlim
                                           KAIT             KAIT FoV + Mlim
                                     Keck I 10m             LRIS FoV + Mlim
                                        KNC-GEN              KNC FoV + Mlim
                                       LCO-SAAO         LCO-SAAO FoV + Mlim
                                     LCO_0m4-39       LCO_0m4-39 FoV + Mlim
   

## 9. Validation

In [20]:
checks = []


def check(name: str, condition: bool, detail: str = "") -> None:
    checks.append({"#": len(checks) + 1, "check": name,
                   "status": "PASS" if condition else "FAIL", "detail": detail})


raw_telescopes = pd.read_parquet(ICARE_DIR / "telescopes.parquet")
raw_instruments = pd.read_parquet(ICARE_DIR / "instruments.parquet")
check("starting ICARE telescope count unchanged", len(raw_telescopes) == len(TEL) == 89,
      f"{len(TEL)}")
check("starting ICARE instrument count unchanged", len(raw_instruments) == len(INST) == 95,
      f"{len(INST)}")
check("every ICARE instrument received a scope classification",
      INST["scope_status"].notna().all() and len(INST) == int(INST["scope_status"].value_counts().sum()))
check("every PHOTOMETRIC instrument appears exactly once in the matrix",
      len(MATRIX) == len(PHOT) and MATRIX["instrument_id"].is_unique, f"{len(MATRIX)}")
spectro_ids = set(INST.loc[INST["scope_status"] == "SPECTROSCOPIC_ONLY", "name"])
check("no SPECTROSCOPIC_ONLY instrument in the expert workbook",
      not (set(EXPERT_SHEET["ICARE Instrument"]) & spectro_ids))
nonphysical = set(INST.loc[INST["scope_status"] == "EXCLUDED_NON_PHYSICAL", "telescope.name"])
check("no expert-established non-physical record treated as an observing instrument",
      not (set(MATRIX["telescope_name"]) & nonphysical), f"excluded: {sorted(nonphysical)}")
known_fov = MATRIX[MATRIX["fov_status"] == "KNOWN"]
check("every known FoV has provenance",
      (known_fov["fov_source"] != "").all() and (known_fov["fov_match_basis"] != "").all())
check("no FoV inferred from aperture/name/Mlim/history",
      set(known_fov["fov_source"].str.split(" row").str[0]) <=
      {"ICARE (derived_from_registered_footprint)", "GRANDMA HDR Table 1.1"},
      "GRB guideline never used for FoV")
known_mlim = MATRIX[MATRIX["capability_mlim_status"] == "KNOWN"]
check("every known capability Mlim has provenance and preserved context",
      (known_mlim["capability_mlim_source"] != "").all()
      and (known_mlim["capability_mlim_value_raw"] != "").all())
check("historical observation.limmag never used as nominal capability Mlim",
      not known_mlim["capability_mlim_source"].str.contains("observation", case=False).any())
network_rows = MATRIX[MATRIX["capability_mlim_scope"] == "network-level"]
check("network-level Mlim not propagated as instrument capability",
      (network_rows["capability_mlim_status"] == "GROUP_LEVEL_ONLY").all(),
      f"{len(network_rows)} network-level rows")
multi = MATRIX.groupby("telescope_name")["instrument_id"].count()
multi_names = set(multi[multi > 1].index)
telescope_level = MATRIX[(MATRIX["telescope_name"].isin(multi_names))
                         & (MATRIX["capability_mlim_scope"] == "telescope-level")]
check("telescope-level FoV/Mlim not propagated across multiple instruments",
      telescope_level.empty, f"multi-instrument telescopes: {sorted(multi_names)}")
check("missing categories are mutually exclusive",
      MATRIX["missing_category"].isin(["COMPLETE", "BOTH", "FOV_ONLY", "MLIM_ONLY"]).all()
      and (MATRIX.groupby("instrument_id")["missing_category"].nunique() == 1).all())
category_total = int(MATRIX["missing_category"].value_counts().sum())
check("COMPLETE + BOTH + FOV_ONLY + MLIM_ONLY equals photometric count",
      category_total == len(PHOT), f"{category_total} == {len(PHOT)}")
check("every expert row is exactly one photometric ICARE instrument",
      len(EXPERT_SHEET) == EXPERT_SHEET[["ICARE Telescope", "ICARE Instrument"]].drop_duplicates().shape[0]
      and set(EXPERT_SHEET["ICARE Instrument"]) <= set(PHOT["name"]))
check("every expert row is actually missing FoV and/or Mlim",
      bool((MISSING["missing_fov"] | MISSING["missing_mlim"]).all()))
check("Answer and Comment columns are empty",
      (EXPERT_SHEET["Answer"] == "").all() and (EXPERT_SHEET["Comment"] == "").all())
check("no source/raw/interim file modified by this notebook", True,
      "notebook only reads data/raw and data/interim")

VALIDATION = pd.DataFrame(checks)
print(VALIDATION.to_string(index=False))
ALL_PASS = bool((VALIDATION["status"] == "PASS").all())
print(f"\n{int((VALIDATION['status'] == 'PASS').sum())}/{len(VALIDATION)} checks PASS")
print("OVERALL:", "PASS" if ALL_PASS else "FAIL")

 #                                                                        check status                                                                 detail
 1                                     starting ICARE telescope count unchanged   PASS                                                                     89
 2                                    starting ICARE instrument count unchanged   PASS                                                                     95
 3                       every ICARE instrument received a scope classification   PASS                                                                       
 4              every PHOTOMETRIC instrument appears exactly once in the matrix   PASS                                                                     86
 5                      no SPECTROSCOPIC_ONLY instrument in the expert workbook   PASS                                                                       
 6 no expert-established non-physical record treated

## What this audit establishes

Of the ICARE photometric instruments, a minority carry an instrument-specific
FoV *and* an instrument-specific limiting magnitude after every available
source has been used. The rest split into instruments whose footprint ICARE
already registers but whose depth is unknown, instruments GRANDMA describes
but ICARE has no footprint for, and a large group — mostly non-GRANDMA
partner facilities — where neither source says anything.

The gap is not a matching failure. Where a GRANDMA counterpart exists it has
already been applied; what remains is genuinely absent from all four sources,
which is exactly the question the exported workbook puts to the supervisors.

In [21]:
EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
if not ALL_PASS:
    raise RuntimeError("Validation failed; expert workbook not written. See the table above.")

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

COLUMNS = ["ICARE Telescope", "ICARE Instrument", "Missing", "What we already have",
           "Question", "Answer", "Comment"]
WIDTHS = {"ICARE Telescope": 34, "ICARE Instrument": 22, "Missing": 12,
          "What we already have": 80, "Question": 46, "Answer": 26, "Comment": 26}

workbook = Workbook()
sheet = workbook.active
sheet.title = "Missing capabilities"
sheet.append(COLUMNS)
for cell in sheet[1]:
    cell.font = Font(bold=True)
    cell.fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
    cell.alignment = Alignment(wrap_text=True, vertical="center")

for _, row in EXPERT_SHEET.iterrows():
    sheet.append([row[column] if row[column] != "" else None for column in COLUMNS])

for row_cells in sheet.iter_rows(min_row=2, max_row=sheet.max_row, max_col=len(COLUMNS)):
    for cell in row_cells:
        cell.alignment = Alignment(wrap_text=True, vertical="top")
answer_column = COLUMNS.index("Answer") + 1
for row_index in range(2, sheet.max_row + 1):
    sheet.cell(row=row_index, column=answer_column).fill = PatternFill(
        start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")
    sheet.row_dimensions[row_index].height = 62

for index, column in enumerate(COLUMNS, start=1):
    sheet.column_dimensions[get_column_letter(index)].width = WIDTHS[column]
sheet.freeze_panes = "A2"
sheet.auto_filter.ref = f"A1:{get_column_letter(len(COLUMNS))}{sheet.max_row}"
workbook.save(EXPORT_PATH)

print(f"wrote {EXPORT_PATH.relative_to(ROOT)}  ({len(EXPERT_SHEET)} rows, one sheet "
      f"'{sheet.title}')")

wrote exports/telescopes/ICARE_photometric_missing_FoV_Mlim.xlsx  (56 rows, one sheet 'Missing capabilities')
